# Session 19 — ESM-2 GPU benchmarks

Runs the GPU-only halves of Session 18's optimization benchmarks and auto-pushes results back to the branch. Three benchmarks land in this session:

1. **ESM-2 batch-size sensitivity** (`scripts/bench_esm2_batch.py`) — batch sizes 8 / 16 / 32 / 64 on 32 synthetic 300-aa sequences. Tells us the optimal batch size for the populate notebook.
2. **ESM-2 150M vs 650M** — embed all 452 Syn3A CDS with both models inline, time wall + measure dim. Decides whether the 150M model is worth switching to (40% of params, 50% of dims).
3. **XGBoost `gpu_hist` vs CPU `hist`** (`scripts/bench_xgboost_treemethod.py`) — bonus benchmark, auto-detects CUDA and overwrites Session 18's published-estimate placeholder with a real number.

**PAT setup (one-time):** open the Colab left sidebar → 🔑 Secrets → add a secret named `GITHUB_PAT` with your token. The auto-push cell pulls it via `userdata.get`. You won't be prompted again on subsequent runs.

Wall-time estimate end-to-end on A100: ~10 minutes (most of it is model downloads on first run).


In [ ]:
# Cell 1 — install GPU-side deps + verify CUDA.
!pip install -q \
    "torch>=2.2" \
    "transformers>=4.40" \
    "biopython>=1.83" \
    "pyarrow>=14" \
    "pandas>=2" \
    "xgboost>=2.0" \
    "scikit-learn>=1.3"

import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {props.total_memory / 1024**3:.1f} GiB")
else:
    raise RuntimeError(
        "This notebook requires a GPU. Runtime menu "
        "-> Change runtime type -> A100 / L4 / T4."
    )

In [ ]:
# Cell 2 — clone (or refresh) the project repo.
# Force-fetch + reset --hard on re-run so we always benchmark the
# branch-tip code, not whatever a previous kernel left in /content.
import os, subprocess

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    print(f"cloning {REPO_URL} (branch {BRANCH})...")
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    print(f"{REPO_DIR} exists; refreshing to origin/{BRANCH}...")
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

%cd /content/cell
import sys
sys.path.insert(0, "/content/cell")
_run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)

In [ ]:
# Cell 3 — prompt for GitHub PAT.
# Uses getpass so the token is masked in the cell input. The PAT
# stays in this kernel's environment for the rest of the session;
# re-run this cell to overwrite it (e.g. if you rotated the token).
import os
import getpass

if os.environ.get("GITHUB_PAT", "").strip():
    print(f"GITHUB_PAT already set in this kernel "
          f"({len(os.environ['GITHUB_PAT'])} chars). "
          f"Re-run this cell to overwrite.")
else:
    pat = getpass.getpass(
        "Paste your GitHub fine-grained PAT (input hidden): "
    ).strip()
    if not pat:
        raise ValueError("empty PAT — cannot push later")
    os.environ["GITHUB_PAT"] = pat
    print(f"GITHUB_PAT set ({len(pat)} chars).")


In [ ]:
# Cell 4 — Bench 1: ESM-2 batch-size scan.
# scripts/bench_esm2_batch.py auto-detects CUDA and writes measurement
# numbers into the existing outputs/bench_esm2_batch_plan.json. The
# 'status' field flips from 'AWAITING_COLAB' to 'MEASURED'.
!python scripts/bench_esm2_batch.py \
    --out outputs/bench_esm2_batch_plan.json

import json
result = json.load(open("outputs/bench_esm2_batch_plan.json"))
print(f"\nstatus: {result['status']}")
if "measurements" in result:
    print("batch-size scan results:")
    for bs, m in result["measurements"].items():
        print(f"  bs={bs:>3s}  wall={m['wall_s']:.2f}s  "
              f"throughput={m['throughput_seq_per_s']:.2f} seq/s")
    print(f"\nbest batch size: {result['best_batch_size']}")

In [ ]:
# Cell 5 — Bench 2: ESM-2 150M vs 650M side-by-side.
# We do the embedding inline (rather than via ESM2Extractor) because
# the extractor hardcodes _MODEL_ID = 'facebook/esm2_t33_650M_UR50D'
# (see bench_esm2_sizes_plan.json's 'blocking_codechange' note). The
# inline path uses transformers directly — same model weights, same
# tokenizer, same fp16 + GPU pattern as the production extractor.
import time, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel

STAGED_DIR = Path(
    "cell_sim/data/Minimal_Cell_ComplexFormation/input_data"
)
STAGED_DIR.mkdir(parents=True, exist_ok=True)
GB_PATH = STAGED_DIR / "syn3A.gb"
if not GB_PATH.exists():
    print("staging syn3A.gb from Luthey-Schulten upstream...")
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Luthey-Schulten-Lab/"
        "Minimal_Cell_ComplexFormation/master/input_data/syn3A.gb",
        GB_PATH,
    )

rec = next(SeqIO.parse(GB_PATH, "genbank"))
rows = []
for f in rec.features:
    if f.type != "CDS":
        continue
    lt = (f.qualifiers.get("locus_tag") or [""])[0]
    tr = (f.qualifiers.get("translation") or [""])[0]
    if lt and tr:
        rows.append({
            "locus_tag": lt,
            "sequence": tr.upper().rstrip("*"),
        })
df = pd.DataFrame(rows)
print(f"{len(df)} CDS with translations")

MODELS = [
    ("esm2_150M", "facebook/esm2_t30_150M_UR50D", 640),
    ("esm2_650M", "facebook/esm2_t33_650M_UR50D", 1280),
]
BATCH_SIZE = 16
DEVICE = "cuda"

results = {}
for name, model_id, embed_dim in MODELS:
    print(f"\n=== {name}  ({model_id}) ===")
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).to(DEVICE).eval()
    model = model.half()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    embeds = np.full((len(df), embed_dim), np.nan, dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(df), BATCH_SIZE):
            chunk = df.iloc[i:i + BATCH_SIZE]
            enc = tok(
                chunk["sequence"].tolist(),
                padding=True, truncation=True, max_length=1022,
                return_tensors="pt",
            ).to(DEVICE)
            out = model(**enc)
            # Mean-pool over residues, mask out pad tokens.
            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (out.last_hidden_state * mask).sum(dim=1)
            pooled = (summed / mask.sum(dim=1)).float().cpu().numpy()
            embeds[i:i + len(chunk)] = pooled
    torch.cuda.synchronize()
    wall = time.perf_counter() - t0
    cols = [f"{name}_dim_{j}" for j in range(embed_dim)]
    feats = pd.DataFrame(embeds, columns=cols)
    feats.index = pd.Index(df["locus_tag"], name="locus_tag")
    parquet_path = Path(f"cell_sim/features/cache/{name}.parquet")
    feats.to_parquet(parquet_path)
    import hashlib
    sha = hashlib.sha256(parquet_path.read_bytes()).hexdigest()
    results[name] = {
        "model_id": model_id,
        "embedding_dim": embed_dim,
        "wall_s": wall,
        "throughput_seq_per_s": len(df) / wall,
        "parquet": str(parquet_path),
        "sha256": sha,
        "rows": len(feats),
    }
    print(f"  wall={wall:.1f}s  throughput={len(df)/wall:.2f} seq/s")
    print(f"  saved -> {parquet_path}  sha={sha[:12]}...")
    del model, tok
    torch.cuda.empty_cache()

speedup = results["esm2_650M"]["wall_s"] / results["esm2_150M"]["wall_s"]
print(f"\n150M is {speedup:.2f}x faster than 650M for the embedding pass")

# Overwrite the plan-only output with measured numbers.
out_path = Path("outputs/bench_esm2_sizes_plan.json")
plan = json.load(open(out_path))
plan["status"] = "MEASURED"
plan["measurements"] = results
plan["inference_speedup_150M_vs_650M"] = speedup
out_path.write_text(json.dumps(plan, indent=2, default=float))
print(f"updated {out_path}")

In [ ]:
# Cell 6 — Bench 3: XGBoost gpu_hist vs CPU hist.
# scripts/bench_xgboost_treemethod.py auto-detects CUDA and runs the
# GPU branch when available. Sandbox-side it only ran the CPU branch
# and stored a published-estimate placeholder; this overwrites it
# with a real number.
!python scripts/bench_xgboost_treemethod.py \
    --out outputs/bench_xgboost_treemethod.json

import json
r = json.load(open("outputs/bench_xgboost_treemethod.json"))
print(f"\nCPU hist:  wall={r['cpu_hist']['wall_s']:.1f}s  "
      f"pooled_mcc={r['cpu_hist']['pooled_mcc']:.4f}")
if isinstance(r.get("gpu_hist"), dict):
    print(f"GPU hist:  wall={r['gpu_hist']['wall_s']:.1f}s  "
          f"pooled_mcc={r['gpu_hist']['pooled_mcc']:.4f}")
    print(f"speedup:   {r['gpu_speedup_factor']:.2f}x")
else:
    print(f"GPU hist:  {r.get('gpu_hist')}")

In [ ]:
# Cell 7 — auto-push results back to the branch.
# Files committed:
#   - outputs/bench_esm2_batch_plan.json (now MEASURED)
#   - outputs/bench_esm2_sizes_plan.json (now MEASURED)
#   - outputs/bench_xgboost_treemethod.json (now has gpu_hist)
#   - cell_sim/features/cache/esm2_150M.parquet (new)
#   - cell_sim/features/cache/esm2_650M.parquet (re-embedded; same
#     model + same input -> sha should match the existing one and
#     the diff is a no-op)
import os, subprocess
from pathlib import Path

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat:
    raise SystemExit(
        "GITHUB_PAT not set in this kernel. Re-run Cell 3 (Colab "
        "Secrets) or, as a one-off, set it inline:\n"
        "    %env GITHUB_PAT=ghp_xxx"
    )

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
COMMIT_MSG = (
    "Session 19: ESM-2 GPU benchmarks (batch-size scan, 150M vs 650M, "
    "XGBoost gpu_hist)"
)

def run(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
run(["git", "config", "user.name", "cell-sim-bot"])

# Auto-discover everything that changed under outputs/ and the cache.
to_add = []
for p in [
    "outputs/bench_esm2_batch_plan.json",
    "outputs/bench_esm2_sizes_plan.json",
    "outputs/bench_xgboost_treemethod.json",
    "cell_sim/features/cache/esm2_150M.parquet",
    "cell_sim/features/cache/esm2_650M.parquet",
]:
    if Path(p).exists() and Path(p).stat().st_size > 0:
        to_add.append(p)

print(f"staging {len(to_add)} files:")
for p in to_add: print(f"  {p}")
run(["git", "add", "-f", *to_add])

status = run(["git", "status", "--porcelain"], check=False)
if not status.stdout.strip():
    print("nothing changed -- benchmarks reproduced byte-identical")
else:
    run(["git", "commit", "-m", COMMIT_MSG])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote, BRANCH])
    print("\npush complete.")

run(["git", "log", "--oneline", "-3"])